In [ ]:
from pathlib import Path
import os
import sys

_here = Path.cwd().resolve()
_repo_root = next(path for path in (_here, *_here.parents) if (path / "configs" / "cfg.py").exists())
os.chdir(_repo_root)
sys.path.insert(0, str(_repo_root))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

import scripts.compare as compare_module
from utils.run_artifacts import load_experiment_context

cfg, run_dir = load_experiment_context("artifacts/runs/20260510_172102_43adfd99")
run_dir = Path(run_dir)

# EV soft-constraint configuration
cfg.env.ev_enabled = True
cfg.model.action_dim = 3
cfg.env.ev_capacity_kwh = (60.0, 60.0, 60.0)
cfg.env.ev_soc_min = 0.10
cfg.env.ev_soc_max = 0.95
cfg.env.ev_arrival_soc = 0.15
cfg.env.ev_departure_soc_req = 0.90
cfg.env.ev_max_charge_kw = (11.0, 11.0, 11.0)
cfg.env.ev_efficiency = 0.95
cfg.env.ev_arrival_step = 72
cfg.env.ev_departure_step = 28
cfg.reward.ev_departure_penalty_weight = 500.0
cfg.reward.ev_soc_regularization_weight = 0.005

display(Markdown("# Compare + EV Soft Constraint"))
display(pd.DataFrame([{
    "run_dir": str(run_dir),
    "ev_enabled": bool(cfg.env.ev_enabled),
    "action_dim": int(cfg.model.action_dim),
    "ev_capacity_kwh": cfg.env.ev_capacity_kwh,
    "ev_arrival_soc": float(cfg.env.ev_arrival_soc),
    "ev_departure_soc_req": float(cfg.env.ev_departure_soc_req),
    "ev_max_charge_kw": cfg.env.ev_max_charge_kw,
    "ev_arrival_step": int(cfg.env.ev_arrival_step),
    "ev_departure_step": int(cfg.env.ev_departure_step),
    "ev_departure_penalty_weight": float(cfg.reward.ev_departure_penalty_weight),
}]))

available_schemes = []
for mode, controller in compare_module.COMPARE_SCHEMES:
    step_path = run_dir / "results" / mode / controller / "record" / "step.parquet"
    if not step_path.exists():
        print(f"Skip missing result: {mode}/{controller}")
        continue
    try:
        columns = pd.read_parquet(step_path, engine="pyarrow").columns
    except Exception as exc:
        print(f"Skip unreadable result: {mode}/{controller}: {exc}")
        continue
    if "ev_charge_total" not in columns:
        print(f"Skip missing result: {mode}/{controller}")
        continue
    available_schemes.append((mode, controller))

if not available_schemes:
    display(Markdown("## No EV soft records found"))
    print("Run the EV soft notebooks under notebooks/notebooks_EV first, and write results to the current run_dir.")
else:
    print("EV soft results included in compare:")
    for mode, controller in available_schemes:
        print(f"- {mode}/{controller}")

    compare_module.COMPARE_SCHEMES = available_schemes
    result = compare_module.compare_records(cfg, run_dir)

    display(result["metrics_df"])
    display(result["economic_table"])
    display(result["safety_table"])

    _plot_specs = [
        ("1. Power balance", "compare_price"),
        ("2. Price", "compare_voltage"),
        ("3. Net load", "compare_net_load"),
        ("4. Battery power and SoC", "compare_battery_soc"),
        ("5. Power balance", "compare_power_balance"),
    ]

    for title, name in _plot_specs:
        print(f"{title} - EV soft constraint")
        if name in result["figures"]:
            display(result["figures"][name])
            plt.close(result["figures"][name])
        else:
            print(f"Figure missing, skipped: {name}")
